# Week 2 - Supervised Learning (Regression & Classification)
## Assignments 1 & 2 + Mini Project 2: House Price Prediction Model

---
### Overview & Learning Objectives
- **Focus**: Model building, training, evaluation, and accuracy testing.
- **Topics Covered**:
  - Linear Regression
  - Logistic Regression
  - Decision Tree, Random Forest
  - Accuracy metrics: Mean Squared Error (MSE), Root Mean Squared Error (RMSE), $R^2$ Score, Confusion Matrix, Classification Report, ROC-AUC.

---
### Structure
1. **Assignment 1**: Build Linear Regression model on housing dataset (predict price).
2. **Assignment 2**: Train Logistic Regression on Titanic dataset for survival prediction (with Decision Tree & Random Forest comparison).
3. **Mini Project 2**: *House Price Prediction Model* (Kaggle House Prices Dataset) - full training, testing, $R^2$ evaluation, and visual plotting.

---
## 1. Assignment 1: Linear Regression on Housing Dataset
**Goal**: Build a Linear Regression model to predict house prices using continuous and categorical features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. Load the housing dataset
df_housing = pd.read_csv('Housing.csv')
print(f'Housing Dataset Shape: {df_housing.shape[0]} rows, {df_housing.shape[1]} columns')
df_housing.head()

In [ ]:
# Check dataset info and null values
df_housing.info()
print('\nMissing values:\n', df_housing.isnull().sum())

In [ ]:
# Preprocessing: Encode binary categorical features (yes/no -> 1/0)
df_h_processed = df_housing.copy()
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df_h_processed[col] = df_h_processed[col].map({'yes': 1, 'no': 0})

# Dummy encode multi-class categorical column
df_h_processed = pd.get_dummies(df_h_processed, columns=['furnishingstatus'], drop_first=True, dtype=int)
df_h_processed.head()

In [ ]:
# Split into Features (X) and Target (y)
X_h = df_h_processed.drop(columns=['price'])
y_h = df_h_processed['price']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)
print(f'Train samples: {len(X_train_h)}, Test samples: {len(X_test_h)}')

# Train Linear Regression model
lr_housing = LinearRegression()
lr_housing.fit(X_train_h, y_train_h)

# Predict on test set
y_pred_h = lr_housing.predict(X_test_h)

# Evaluate performance
r2 = r2_score(y_test_h, y_pred_h)
mse = mean_squared_error(y_test_h, y_pred_h)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_h, y_pred_h)

print(f'R-squared (R2) Score          : {r2:.4f}')
print(f'Mean Squared Error (MSE)      : {mse:,.2f}')
print(f'Root Mean Squared Error (RMSE): ${rmse:,.2f}')
print(f'Mean Absolute Error (MAE)     : ${mae:,.2f}')

In [ ]:
# Feature Importance: Inspect Regression Coefficients
coef_df = pd.DataFrame({
    'Feature': X_h.columns,
    'Coefficient ($)': lr_housing.coef_
}).sort_values(by='Coefficient ($)', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=coef_df, x='Coefficient ($)', y='Feature', palette='viridis')
plt.title('Impact of Features on Housing Price (Coefficients)', fontsize=13, fontweight='bold')
plt.xlabel('Coefficient Value ($)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
coef_df

---
## 2. Assignment 2: Logistic Regression on Titanic Dataset
**Goal**: Train a Logistic Regression classifier for survival prediction, compute accuracy, confusion matrix, and compare with Decision Tree and Random Forest models.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

# Load Titanic dataset
df_titanic = pd.read_csv('Titanic-Dataset.csv')
print(f'Titanic Dataset Shape: {df_titanic.shape}')
df_titanic.head(3)

In [ ]:
# Preprocessing and handling missing values
df_t_clean = df_titanic.copy()
df_t_clean['Age'] = df_t_clean['Age'].fillna(df_t_clean['Age'].median())
df_t_clean['Embarked'] = df_t_clean['Embarked'].fillna(df_t_clean['Embarked'].mode()[0])

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X_t = df_t_clean[features].copy()
y_t = df_t_clean['Survived']

X_t['Sex'] = X_t['Sex'].map({'male': 0, 'female': 1})
X_t = pd.get_dummies(X_t, columns=['Embarked'], drop_first=True, dtype=int)

# Train/Test Split (stratified)
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t
)

# Feature Scaling
scaler = StandardScaler()
X_train_t_scaled = scaler.fit_transform(X_train_t)
X_test_t_scaled = scaler.transform(X_test_t)

# Train Logistic Regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_t_scaled, y_train_t)

# Predict
y_pred_t = log_reg.predict(X_test_t_scaled)
y_pred_proba_t = log_reg.predict_proba(X_test_t_scaled)[:, 1]

acc = accuracy_score(y_test_t, y_pred_t)
roc_auc = roc_auc_score(y_test_t, y_pred_proba_t)
print(f'Logistic Regression Accuracy: {acc * 100:.2f}%')
print(f'Logistic Regression ROC-AUC : {roc_auc:.4f}')

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test_t, y_pred_t)
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Perished (0)', 'Survived (1)'],
            yticklabels=['Perished (0)', 'Survived (1)'])
plt.title('Confusion Matrix - Logistic Regression', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print('\nClassification Report:\n')
print(classification_report(y_test_t, y_pred_t, target_names=['Did Not Survive (0)', 'Survived (1)']))

In [ ]:
# Comparison with Decision Tree & Random Forest (Topics Covered in Week 2)
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_t, y_train_t)
dt_acc = accuracy_score(y_test_t, dt.predict(X_test_t))

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train_t, y_train_t)
rf_acc = accuracy_score(y_test_t, rf.predict(X_test_t))

comp_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree (max_depth=4)', 'Random Forest (100 trees)'],
    'Accuracy': [f'{acc*100:.2f}%', f'{dt_acc*100:.2f}%', f'{rf_acc*100:.2f}%'],
    'ROC-AUC': [f'{roc_auc:.4f}', f"{roc_auc_score(y_test_t, dt.predict_proba(X_test_t)[:, 1]):.4f}", f"{roc_auc_score(y_test_t, rf.predict_proba(X_test_t)[:, 1]):.4f}"]
})
comp_df

---
## 3. Mini Project 2: House Price Prediction Model
**Dataset**: Kaggle – House Prices Dataset (`Housing.csv`)
**Tasks**:
- Train Linear Regression model
- Predict house prices
- Evaluate $R^2$ score and regression metrics
- Plot predicted vs actual values & residuals

In [ ]:
# Complete Mini Project Model Training and Prediction
project_model = LinearRegression()
project_model.fit(X_train_h, y_train_h)
predictions = project_model.predict(X_test_h)

# Evaluate
proj_r2 = r2_score(y_test_h, predictions)
proj_rmse = np.sqrt(mean_squared_error(y_test_h, predictions))
proj_mae = mean_absolute_error(y_test_h, predictions)

print('=== Mini Project 2 Evaluation Results ===')
print(f'R² Score: {proj_r2:.4f} ({proj_r2*100:.2f}% variance explained)')
print(f'RMSE    : ${proj_rmse:,.2f}')
print(f'MAE     : ${proj_mae:,.2f}')

In [ ]:
# Plot Predicted vs Actual Values and Residuals
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Predicted vs Actual
sns.scatterplot(x=y_test_h / 1e6, y=predictions / 1e6, ax=axes[0], color='#1f77b4', alpha=0.8, s=60, edgecolor='black')
min_p = min(y_test_h.min(), predictions.min()) / 1e6
max_p = max(y_test_h.max(), predictions.max()) / 1e6
axes[0].plot([min_p, max_p], [min_p, max_p], color='#d62728', linestyle='--', lw=2, label='Perfect Fit (y = x)')
axes[0].set_title(f'Predicted vs Actual House Prices (R² = {proj_r2:.4f})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Price (Millions $)', fontsize=11)
axes[0].set_ylabel('Predicted Price (Millions $)', fontsize=11)
axes[0].legend()

# 2. Residuals Distribution
residuals = y_test_h - predictions
sns.histplot(residuals / 1e6, kde=True, ax=axes[1], color='#9467bd', bins=20)
axes[1].axvline(0, color='red', linestyle='--', lw=2, label='Zero Error')
axes[1].set_title(f'Residuals Distribution (RMSE = ${proj_rmse/1e6:.2f}M)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted) [Millions $]', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=300)
plt.show()

In [ ]:
# Export Test Predictions to CSV
pred_export = X_test_h.copy()
pred_export['actual_price'] = y_test_h.values
pred_export['predicted_price'] = np.round(predictions, 2)
pred_export['residual'] = np.round(residuals.values, 2)
pred_export.to_csv('housing_predictions.csv', index=False)
print(f'Exported test predictions to housing_predictions.csv (Rows: {len(pred_export)})')
pred_export.head()